# Prerequisites: .env file

To work with DataRobot services, you have to load in your environment DataRobot credentials. We suggest making an `.env` file of the following structure:
```
DATAROBOT_ENDPOINT=
DATAROBOT_API_TOKEN=
```

In [5]:
# -------------------
# 0. Load dotenv and install dependencies 
# -------------------
%load_ext dotenv
%dotenv

!uv pip install "datarobot-genai[langgraph]>=0.15.3"
!uv tool install git+https://github.com/carsongee/get-datarobot-llms.git

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv
Audited 1 package in 80ms
Resolved 18 packages in 21ms                                         
Audited 18 packages in 1ms
Installed 1 executable: dr-get-llms


# Agent and its components
## LLM

DataRobot LLM Gateway provides access to a variety of LLM options. Before you start with your action, explore which options are available.

> May take some time depending on your network connection

In [7]:
# -------------------
# 1. List LLMs we have
# -------------------
!dr get-llms

]11;?\]10;?\]11;?\🔌 Running plugin: get-llms
etching models from DataRobot...
DataRobot Available Models

Llm Id                                          Model                                             
----------------------------------------------  --------------------------------------------------
anthropic-1p-claude-opus-4-1                    anthropic/claude-opus-4-1-20250805                
anthropic-1p-claude-opus-4                      anthropic/claude-opus-4-20250514                  
anthropic-1p-claude-opus-4-5                    anthropic/claude-opus-4-5-20251101                
anthropic-1p-claude-sonnet-4                    anthropic/claude-sonnet-4-20250514                
anthropic-1p-claude-sonnet-4-5                  anthropic/claude-sonnet-4-5-20250929              
anthropic-1p-claude-sonnet-4-6                  anthropic/claude-sonnet-4-6                       
azure-openai-gpt-4-o                            azure/gpt-4o-2024-11-20                         

`datarobot_genai` offers a unified layer of LLM adaptors for any framework you choose and any option of LLM consumption. For LLM GW, import function `get_datarobot_gateway_llm` from `llm` module of the framework you select.

In [8]:
# -------------------
# 1. Instantiate an LLM object
# -------------------
from datarobot_genai.langgraph.llm import get_datarobot_gateway_llm

llm = get_datarobot_gateway_llm("azure/gpt-5-nano-2025-08-07")

## Tools
You may choose to equip your agent with tools. DataRobot offers a selection of functions which can be used as tools in a module `drtools`.

> Note that `drtools` is currently undergoing refactoring, not all tools will be available to use standalone (outside of `drmcp`).

In [9]:
# -------------------
# 2. Tools
# -------------------
from datarobot_genai.drtools.calculator import calculator
from langchain.tools import tool

tools = [tool(calculator)]

## Agent

The agent definition has little to none difference from how you define an agent in a native framework. You use the same primitives (crew, graph, workflow) which makes it simple to adopt existing use-cases.

In [10]:
from typing import TypedDict
from langgraph.graph import StateGraph, END

from langchain.agents import create_agent

from langchain_litellm.chat_models import ChatLiteLLM
from langchain_core.prompts import ChatPromptTemplate
from langgraph.graph import MessagesState
from langgraph.graph import END
from langgraph.graph import START

# -------------------
# 3.1. Agent prompt
# -------------------
prompt_template = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful assistant. {chat_history}"),
        ("user", "{topic}"),
    ]
)

def graph_factory(*args):
    # -------------------
    # 3.2. ReAct agents as nodes
    # -------------------
    research_agent = create_agent(llm, tools)
    writer_agent = create_agent(llm, [])
    
    # -------------------
    # 3.3. Graph
    # -------------------
    graph = StateGraph(MessagesState)
    graph.add_node("research", research_agent)
    graph.add_node("writer", writer_agent)
    graph.add_edge(START, "research")
    graph.add_edge("research", "writer")
    graph.add_edge("writer", END)
    return graph

# Convert your agent into a DataRobot agent
For every framework DataRobot supports, there is function which lets you convert this framework primitives into an agent class to be used in DataRobot.

In [11]:
# -------------------
# 4. Convert agent
# -------------------
from datarobot_genai.langgraph.agent import datarobot_agent_class_from_langgraph

MyAgent = datarobot_agent_class_from_langgraph(graph_factory, prompt_template)

# Action!

In [12]:
# -------------------
# 5. Action!
# -------------------
from ag_ui.core import UserMessage
from ag_ui.core import RunAgentInput
import uuid

agent = MyAgent()

run_input = RunAgentInput(
    thread_id=str(uuid.uuid4()),
    run_id=str(uuid.uuid4()),
    messages=[UserMessage(id=str(uuid.uuid4()), role="user", content="Calculate 2+2 and outputn result immediately")],
    state=[],
    tools=[],
    context=[],
    forwardedProps=None
)

async for event, _interactions, usage in agent.invoke(run_input):
    print(event)

type=<EventType.RUN_STARTED: 'RUN_STARTED'> timestamp=None raw_event=None thread_id='1aa0349f-9bef-455e-b941-6afbadc4259c' run_id='1f29f2cd-9e53-4ff6-bd8b-20d05aeee409' parent_run_id=None input=None
[values] {'messages': [SystemMessage(content='You are a helpful assistant. ', additional_kwargs={}, response_metadata={}, id='8250ee5d-30b1-49aa-b21a-f479ab84dc12'), HumanMessage(content='Calculate 2+2 and outputn result immediately', additional_kwargs={}, response_metadata={}, id='02415a13-1560-4683-ab08-0306472b67bd')]}
[values] [graph=('research:bf711803-3150-3506-3b01-4e207cdd7f4b',)] {'messages': [SystemMessage(content='You are a helpful assistant. ', additional_kwargs={}, response_metadata={}, id='8250ee5d-30b1-49aa-b21a-f479ab84dc12'), HumanMessage(content='Calculate 2+2 and outputn result immediately', additional_kwargs={}, response_metadata={}, id='02415a13-1560-4683-ab08-0306472b67bd')]}
type=<EventType.TOOL_CALL_START: 'TOOL_CALL_START'> timestamp=None raw_event=None tool_call_id